# RAGAS in Practice — Fundamentals + Real Evaluation
Read `concept_notes.md` and `diagrams.md` first.

This worksheet has **two halves**:
1. **Fundamentals (runs offline now):** we reimplement Faithfulness and Answer Relevancy *from scratch* with a mock judge, so you understand what RAGAS does internally — not just call a black box.
2. **Real RAGAS (wired to your in-house models):** the actual `ragas` library, with the OpenAI default swapped for your in-house chat + Jina embeddings. This part needs your live endpoints, so it's written to run in your environment, not here.

## PART 1 — Fundamentals: reimplement the metrics from scratch
The best way to trust a metric is to build a toy version of it. These run with a mock judge (no API), so you can see the *mechanics*.

In [ ]:
# A mock "judge LLM" and mock embedder so Part 1 runs with no API / no network.
import numpy as np, re

def mock_judge_supported(claim, context):
    """Pretend-LLM: a claim is 'supported' if most of its content words appear in
    the context. Real RAGAS uses an actual LLM for this NLI-style check — this is
    just to show the SHAPE of the computation."""
    claim_words = set(re.findall(r"[a-z0-9]+", claim.lower()))
    ctx_words = set(re.findall(r"[a-z0-9]+", context.lower()))
    stop = {"the","a","is","are","to","of","and","in","by","it","within"}
    claim_words -= stop
    if not claim_words:
        return True
    overlap = len(claim_words & ctx_words) / len(claim_words)
    return overlap >= 0.6

def mock_embed(text):
    v = np.zeros(128)
    for w in re.findall(r"[a-z0-9]+", text.lower()):
        v[abs(hash(w)) % 128] += 1
    n = np.linalg.norm(v); return v/n if n else v

print("Mock judge + embedder ready (Part 1 only).")

### 1a. Faithfulness from scratch
Decompose the answer into atomic claims, check each against the context, score = supported / total. (See diagrams.md section 3.)

In [ ]:
def split_claims(answer):
    # crude claim splitter: one claim per sentence/clause. Real RAGAS uses an LLM.
    parts = re.split(r"[.;]| and ", answer)
    return [p.strip() for p in parts if p.strip()]

def faithfulness(answer, context):
    claims = split_claims(answer)
    if not claims:
        return 1.0, []
    verdicts = [(c, mock_judge_supported(c, context)) for c in claims]
    score = sum(v for _, v in verdicts) / len(verdicts)
    return score, verdicts

context = "Refunds are processed in 5 business days to the original payment method."

# A faithful answer
ans_good = "The refund takes 5 business days and goes to the original payment method."
# A hallucinating answer
ans_bad = "The refund takes 5 business days and is sent by cheque."

for label, ans in [("faithful", ans_good), ("hallucinating", ans_bad)]:
    score, verdicts = faithfulness(ans, context)
    print(f"\n{label}: faithfulness = {score:.2f}")
    for claim, ok in verdicts:
        print(f"   [{'SUPPORTED' if ok else 'NOT SUPPORTED'}] {claim}")

**Observe:** the hallucinating answer scores lower because the 'cheque' claim isn't in the context. This is exactly RAGAS's faithfulness logic — the only difference is RAGAS uses a real LLM to split claims and judge support, which handles paraphrase and inference far better than this word-overlap mock.

### 1b. Answer Relevancy from scratch
Generate questions *from the answer*, embed them, measure similarity to the original question. On-topic answers reverse-generate similar questions.

*(Note: a crude bag-of-words mock is unreliable for semantic similarity, so here we use a small shared vocabulary and word-overlap that rewards literal word reuse — enough to show the mechanic. A real embedding model captures meaning far better; that's why real RAGAS needs one.)*

In [ ]:
def word_overlap_sim(a, b):
    """Jaccard-style overlap — crude but directionally correct for this demo."""
    wa = set(re.findall(r"[a-z]+", a.lower()))
    wb = set(re.findall(r"[a-z]+", b.lower()))
    stop = {"how","what","the","is","a","do","i","to","of","will","you","are","my","for"}
    wa -= stop; wb -= stop
    if not wa or not wb:
        return 0.0
    return len(wa & wb) / len(wa | wb)

def answer_relevancy(question, generated_questions):
    sims = [word_overlap_sim(question, gq) for gq in generated_questions]
    return np.mean(sims), sims

question = "How long do refunds take?"

# On-topic answer reverse-generates questions that reuse 'refund'/'take'/'long':
relevant_gen = ["How long is the refund time?", "How many days do refunds take?",
                "How long until a refund arrives?"]
# Evasive answer reverse-generates questions about unrelated topics:
irrelevant_gen = ["What payment methods exist?", "Where is your office located?",
                  "How do I contact support?"]

for label, gen in [("relevant answer", relevant_gen), ("evasive answer", irrelevant_gen)]:
    score, sims = answer_relevancy(question, gen)
    print(f"{label}: answer_relevancy = {score:.3f}  (per-question sims: {[round(s,2) for s in sims]})")

**Observe:** the relevant answer's reverse-generated questions resemble the original, so its relevancy score is higher. This is why Answer Relevancy needs an *embedding* model (to measure question similarity), while Faithfulness needs a *judge* LLM (to check claim support). Different metric, different machinery.

### 1c. Context Precision & Recall from scratch

In [ ]:
def context_precision(retrieved_relevance):
    """retrieved_relevance: list of 1/0 marking whether each retrieved chunk (in
    rank order) is relevant. Precision@k averaged over the relevant positions —
    rewards putting relevant chunks HIGH."""
    hits, cum = 0, 0.0
    for i, rel in enumerate(retrieved_relevance, 1):
        if rel:
            hits += 1
            cum += hits / i          # precision at this position
    return cum / hits if hits else 0.0

def context_recall(needed_facts_found):
    """needed_facts_found: list of 1/0 marking whether each fact required to answer
    was present in the retrieved context. Recall = fraction found."""
    return sum(needed_facts_found) / len(needed_facts_found) if needed_facts_found else 0.0

# retrieved 4 chunks; chunks 1 and 2 relevant, 3 and 4 not -> good ranking
print("Context precision (relevant ranked high):", round(context_precision([1,1,0,0]), 3))
# same relevant count but ranked LOW -> worse precision
print("Context precision (relevant ranked low): ", round(context_precision([0,0,1,1]), 3))

# answer needed 3 facts; retrieval covered 2 of them
print("Context recall (2 of 3 needed facts found):", round(context_recall([1,1,0]), 3))

**Observe:** same number of relevant chunks, but precision is higher when they're ranked near the top — that ranking-awareness is what separates context precision from plain precision@k. Recall doesn't care about rank, only coverage.

---
## PART 2 — Real RAGAS, wired to your in-house models
Everything below uses the actual `ragas` library. It makes real LLM judge calls, so **run this in your environment** (with your in-house endpoints reachable), not in this offline notebook. The critical part is swapping RAGAS's OpenAI default for your in-house models.

### 2a. Install (pin to the 0.2.x line for this API)

In [ ]:
# Run once in your chroma_lab_env:
# pip install "ragas>=0.2,<0.3" datasets langchain langchain-openai
#
# (langchain-openai is pulled in as a dependency even though you won't call OpenAI —
#  RAGAS uses its wrapper classes internally.)
print("See comment above for the install command.")

### 2b. Point RAGAS at your in-house models (THE key step for your org)
This is what replaces OpenAI. `get_chat_model()` and `InHouseEmbeddings` come from your corrected `inhouse_wrappers.py`.

In [ ]:
# --- THIS CELL NEEDS YOUR LIVE ENDPOINTS ---
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from inhouse_wrappers import get_chat_model, InHouseEmbeddings
from inhouse_llm import MODEL_QWEN3_30B   # a stronger model makes a better judge

# Wrap your in-house chat model as the RAGAS judge
evaluator_llm = LangchainLLMWrapper(get_chat_model(model=MODEL_QWEN3_30B, max_tokens=1000))

# Wrap your in-house Jina embeddings (needed for answer relevancy)
evaluator_embeddings = LangchainEmbeddingsWrapper(InHouseEmbeddings())

print("RAGAS will now use your in-house models as judge + embedder — no OpenAI calls.")

### 2c. Build an evaluation dataset
Each sample = one question your RAG system answered, with the answer it gave, the contexts it retrieved, and (optionally) a reference answer for recall/correctness.

In [ ]:
from ragas import EvaluationDataset
from ragas.dataset_schema import SingleTurnSample

samples = [
    SingleTurnSample(
        user_input="How long do refunds take?",
        response="Refunds are processed within 5 business days to the original payment method.",
        retrieved_contexts=[
            "Refunds are processed within 5 business days to the original payment method once the item is received.",
            "To request a refund, open a ticket with your order number.",
        ],
        reference="Refunds take 5 business days and go to the original payment method.",
    ),
    SingleTurnSample(
        user_input="What is the API rate limit?",
        response="The default API rate limit is 100 requests per minute per key.",
        retrieved_contexts=[
            "The default API rate limit is 100 requests per minute per key. Exceeding it returns HTTP 429.",
        ],
        reference="100 requests per minute per key.",
    ),
    # A deliberately BAD sample: answer hallucinates a detail not in context
    SingleTurnSample(
        user_input="How do I reset my password?",
        response="Use the 'Forgot password' link; the reset code is valid for 24 hours.",  # context says 30 minutes!
        retrieved_contexts=[
            "To reset your password, use the 'Forgot password' link; a reset email is valid for 30 minutes.",
        ],
        reference="Use the forgot-password link; the reset email is valid for 30 minutes.",
    ),
]
dataset = EvaluationDataset(samples=samples)
print(f"Built dataset with {len(samples)} samples.")

### 2d. Run the evaluation with the four core metrics

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    Faithfulness, ResponseRelevancy, LLMContextPrecisionWithReference, LLMContextRecall,
)

# Instantiate the four core metrics (0.2.x class-based names)
metrics = [
    Faithfulness(),
    ResponseRelevancy(),                       # aka answer relevancy
    LLMContextPrecisionWithReference(),
    LLMContextRecall(),
]

result = evaluate(
    dataset=dataset,
    metrics=metrics,
    llm=evaluator_llm,               # <-- your in-house judge
    embeddings=evaluator_embeddings, # <-- your in-house Jina
)

print(result)          # aggregate scores
df = result.to_pandas() # per-sample scores
df

**What to look for in the output:**
- The password sample should score **low on faithfulness** — the answer says '24 hours' but the context says '30 minutes' (a planted hallucination).
- The refund and rate-limit samples should score high across the board.
- `result.to_pandas()` gives you per-sample scores so you can find exactly which question failed which metric — that's how you debug, not just an average.

### 2e. The measure-improve loop (how you'd actually use this)

In [ ]:
# Pseudocode for the loop RAGAS is FOR:
#
# baseline = evaluate(dataset, metrics, llm=evaluator_llm, embeddings=evaluator_embeddings)
#
# ... change ONE thing (chunk size, k, re-ranker, prompt) ...
#
# candidate = evaluate(dataset, metrics, llm=evaluator_llm, embeddings=evaluator_embeddings)
#
# compare candidate vs baseline PER METRIC:
#   faithfulness up but answer_relevancy down? -> your change grounded better but
#     retrieved less on-point context. Net win depends on your priorities.
#
# NEVER ship a pipeline change without re-running this on a FIXED eval set.
print("See comment — this is the core workflow RAGAS enables.")

### Version note
Class names shifted across RAGAS versions:
- **0.2.x** (this worksheet): `Faithfulness()`, `ResponseRelevancy()`, `LLMContextPrecisionWithReference()`, `LLMContextRecall()`, passed as instances.
- **0.1.x**: lowercase module-level objects `faithfulness`, `answer_relevancy`, `context_precision`, `context_recall`, and a HuggingFace `Dataset` instead of `EvaluationDataset`.
- **0.3.x/0.4.x**: newer `llm_factory` + experiment API; the four metrics still exist but setup differs.

If an import fails, check your installed version with `import ragas; print(ragas.__version__)` and match the names to that version's docs. The *concepts* (Part 1) never change — only the library's function names do.

---
## Teaser exercise (ties to concept_notes.md)
Reproduce the concept-notes teaser: construct a sample that scores **high faithfulness but low answer relevancy** (a factually-grounded answer that doesn't address the question). For example, ask 'How do I cancel my subscription?' but feed a retrieved context + answer about *refund timing* — true statements, wrong topic. Confirm RAGAS gives high faithfulness (the answer IS grounded in the provided context) but low answer relevancy (it doesn't address cancellation). Then explain, in one sentence, why this points at a **retrieval** bug, not a generation bug.